In [0]:
class streamWC():
    def __init__(self):
        self.data_path='/Volumes/cat_spark_streaming/wcnt/sample_datasets/test_datasets/'
        self.catalog='cat_spark_streaming'
        self.schema1='wcnt'
        self.checkpoint_name='/Volumes/cat_spark_streaming/wcnt/locationforcheckpoint/checkpoint_data/wordcount/'

    def getRawData(self):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        print('\tFetching raw data...',end='')
        df=(spark.readStream
                .format('text') 
                .option('multiLine', True) 
                .option('lineSep', '.') 
                .load(self.data_path))
        raw_words=df.select(explode(split(col('value')," ")).alias('word'))
        print('completed')    
        return raw_words
    
    def getQualityData(self, df):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        df=df.select((split(col('word'),',')[0]).alias('word'))
        return (df.select(lower(trim(col('word'))).alias('word')) 
                    .where("word is not null") 
                    .where("word rlike '[a-z]'"))

    def getWordCount(self,df):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        return (df 
                .groupBy('word') 
                .agg(count(col('word')).alias('wordCount')))
        
    def overWriteWordCount(self,df):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        print(f'\tSaving table...',end='')
        
        query=(df.writeStream
                    .format('delta')
                    .outputMode('complete')
                    .option('checkpointLocation', '/Volumes/cat_spark_streaming/wcnt/locationforcheckpoint/checkpoint_data/wordcount/')
                    .trigger(availableNow=True)
                    .toTable(f'{self.catalog}.{self.schema1}.wordCountStream'))
        print(f'\tSaved table at location{self.catalog}.{self.schema1}.wordCountStream')
        query.awaitTermination()
        return query

    
    def wordCount(self):
        print('\tStarting Word Count Stream...')
        rawDf=self.getRawData()
        qualifiedDf=self.getQualityData(rawDf)
        wordCountDf=self.getWordCount(qualifiedDf)
        query=self.overWriteWordCount(wordCountDf)
        #query.awaitTermination()
        print('\tStarting Word Count Stream...Completed')

In [0]:
#swc=streamWC()
#swc.wordCount()

	Starting Word Count Stream...
	Fetching raw data...completed
	Saving table...	Saved table at locationcat_spark_streaming.wcnt.wordCountStream


---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-8236643497320338>, line 2
      1 swc=streamWC()
----> 2 swc.wordCount()

File <command-7642940362605965>, line 53, in streamWC.wordCount(self)
     51 wordCountDf=self.getWordCount(qualifiedDf)
     52 query=self.overWriteWordCount(wordCountDf)
---> 53 query.awaitTermination()
     54 print('\tStarting Word Count Stream...Completed')

AttributeError: 'NoneType' object has no attribute 'awaitTermination'

In [0]:


#select * from cat_spark_streaming.wcnt.wordCountStream

word,wordCount
